<a href="https://colab.research.google.com/github/inoue0426/llm-tuning-playground/blob/main/notebooks/03_pharma_dpo_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/inoue0426/llm-tuning-playground/blob/main/notebooks/03_pharma_dpo_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 - DPO on a real pharmaceutical preference dataset

This notebook uses the public `ThakrePranjal/pharma-preference-dataset` and applies LoRA + DPO to Qwen2.5-0.5B-Instruct.

The goal is educational: move from toy preference pairs in Notebook 02 to a small real-domain preference dataset. This is not clinical validation.


## Pipeline

`public pharma preference data -> conversational format -> Qwen + LoRA -> DPO -> held-out evaluation`

The dataset is small, so the training run is intentionally lightweight.


In [1]:
!pip -q install -U \
    "transformers>=4.55,<5" \
    "datasets>=3.6,<5" \
    "peft>=0.17,<1" \
    "trl>=0.21,<1" \
    "accelerate>=1.10,<2" \
    "bitsandbytes>=0.46,<1" \
    "torchao>=0.16,<1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 138.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 50.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import torch
import transformers, datasets, peft, trl

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before running this notebook.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


PyTorch: 2.11.0+cu128
Transformers: 4.57.6
Datasets: 4.8.5
PEFT: 0.20.0
TRL: 0.29.1
CUDA available: True
GPU: NVIDIA L4
VRAM (GB): 22.0


## 1. Load the pharmaceutical preference dataset

Each example is expected to contain `prompt`, `chosen`, and `rejected`.


In [3]:
from datasets import load_dataset

RAW_DATASET = "ThakrePranjal/pharma-preference-dataset"
raw = load_dataset(RAW_DATASET, split="train")
print(raw)
print(raw.column_names)
print(raw[0])


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pharma_preference_dataset.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/48 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
    num_rows: 48
})
['prompt', 'chosen', 'rejected', 'source_page', 'topic']
{'prompt': '### Instruction:\nExplain the primary mechanism of action of metformin.\n\n### Response:\n', 'chosen': 'Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.', 'rejected': 'Metformin mainly works by increasing insulin secretion from the pancreas, and kidney function is usually not very relevant. Its side effects are generally not important unless the patient feels very sick.', 'source_page': 1, 'topic': 'Metformin pharmacology'}


## 2. Convert to TRL conversational preference format

Qwen is an instruction/chat model. We therefore represent the prompt as a user message and the two answers as assistant messages. TRL can then apply Qwen's chat template consistently during DPO preprocessing.


In [4]:
from datasets import Dataset

def normalize_prompt(text):
    text = str(text)
    if "### Instruction:" in text:
        text = text.split("### Instruction:", 1)[1]
    if "### Response:" in text:
        text = text.split("### Response:", 1)[0]
    return text.strip()

rows = []
for row in raw:
    rows.append({
        "prompt": [{"role": "user", "content": normalize_prompt(row["prompt"])}],
        "chosen": [{"role": "assistant", "content": str(row["chosen"]).strip()}],
        "rejected": [{"role": "assistant", "content": str(row["rejected"]).strip()}],
    })

dataset = Dataset.from_list(rows)
split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print("Total:", len(dataset))
print("Train:", len(train_dataset))
print("Eval:", len(eval_dataset))
print("Example:", train_dataset[0])


Total: 48
Train: 38
Eval: 10
Example: {'prompt': [{'role': 'user', 'content': 'What is familial hypercholesterolemia and why is early management important?'}], 'chosen': [{'role': 'assistant', 'content': 'Familial hypercholesterolemia is a genetic condition characterized by elevated LDL-C from a young age and increased lifetime risk of premature cardiovascular disease. Early diagnosis, aggressive lipid management, and family screening are important because risk begins early and can accumulate over time.'}], 'rejected': [{'role': 'assistant', 'content': 'Atorvastatin and ezetimibe are basically the same type of cholesterol medicine and both mainly work in the stomach. Their combination is useful because taking two tablets is always stronger than one.'}]}


## 3. Load Qwen2.5-0.5B-Instruct

We keep the same base model as Notebooks 01 and 02 so the preference dataset is the main change.


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype)
model = model.cuda()

print("Has chat template:", tokenizer.chat_template is not None)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Has chat template: True


## 4. Baseline generation on a held-out prompt


In [6]:
def generate(model, question, max_new_tokens=120):
    messages = [{"role": "user", "content": question}]
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    encoded = {k: v.to(model.device) for k, v in encoded.items()}
    with torch.no_grad():
        output = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

TEST_INDEX = 0
test_prompt = eval_dataset[TEST_INDEX]["prompt"][0]["content"]
print("QUESTION:", test_prompt)
print()
print("BEFORE DPO:")
print(generate(model, test_prompt))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUESTION: Define pharmacovigilance.

BEFORE DPO:
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Define pharmacovigilance.
assistant
Pharmacovigilance is the systematic monitoring of adverse drug reactions (ADRs) and other potential safety issues related to the use of medicines. It involves several key components:

1. **Adverse Drug Reaction Monitoring**: This includes collecting data on ADRs from various sources such as clinical trials, registries, and surveillance systems.

2. **Risk Assessment**: Using this data, risk assessments are conducted to determine the likelihood and severity of each ADR event.

3. **Reporting and Reporting Mechanisms**: Adverse drug reaction reports are collected and analyzed through reporting mechanisms like the European Medicines


## 5. Run DPO with LoRA

The base model is frozen and only a low-rank adapter is trained. `max_prompt_length` is intentionally omitted because the installed TRL 0.29.x API does not accept it in `DPOConfig`.


In [7]:
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

dpo_args = DPOConfig(
    output_dir="./outputs/pharma-dpo",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=1e-5,
    beta=0.1,
    max_length=512,
    logging_steps=2,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
)

trainer = DPOTrainer(
    model=model,
    args=dpo_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.train()


Tokenizing train dataset:   0%|          | 0/38 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
2,0.692800
4,0.685500
6,0.682500
8,0.670100
10,0.665600
12,0.662400
14,0.658700
16,0.633300
18,0.640600
20,0.640400


TrainOutput(global_step=25, training_loss=0.6562957167625427, metrics={'train_runtime': 34.8843, 'train_samples_per_second': 5.447, 'train_steps_per_second': 0.717, 'total_flos': 80959918752768.0, 'train_loss': 0.6562957167625427})

## 6. Evaluate preference behavior

DPO evaluation metrics such as chosen/rejected rewards and preference accuracy are more informative than inspecting a single generated answer.


In [8]:
metrics = trainer.evaluate()
print("Evaluation metrics:")
for key, value in metrics.items():
    if isinstance(value, (int, float)):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")


Evaluation metrics:
eval_loss: 0.6496
eval_runtime: 0.4569
eval_samples_per_second: 21.8860
eval_steps_per_second: 4.3770


In [9]:
model.eval()
print()
print("AFTER DPO:")
print(generate(model, test_prompt))



AFTER DPO:
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Define pharmacovigilance.
assistant
Pharmacovigilance is the systematic monitoring of adverse drug reactions (ADRs) and other potential safety issues related to the use of medicines. It involves several key components:

1. **Adverse Drug Reaction Monitoring**: This includes collecting data on ADRs from various sources such as patient reports, clinical trials, registries, and surveillance systems.

2. **Risk Assessment**: Using this data, healthcare providers can assess the risk associated with each medication or treatment. This helps in prioritizing which medications should be monitored more closely for safety.

3. **Reporting Mechanisms**: Establishing clear reporting


## 7. Inspect several held-out prompts

Read the outputs critically. Fluent text is not evidence of factual correctness.


In [10]:
for i in range(min(5, len(eval_dataset))):
    question = eval_dataset[i]["prompt"][0]["content"]
    print("=" * 80)
    print("QUESTION:", question)
    print(generate(model, question, max_new_tokens=100))


QUESTION: Define pharmacovigilance.
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Define pharmacovigilance.
assistant
Pharmacovigilance is the systematic monitoring of adverse drug reactions (ADRs) and other potential safety issues related to the use of medicines. It involves several key components:

1. **Adverse Drug Reaction Monitoring**: This includes collecting data on ADRs from various sources such as patient reports, clinical trials, registries, and surveillance systems.

2. **Risk Assessment**: Using this data, healthcare providers can assess the risk associated with each medication or treatment. This helps in prioritizing which
QUESTION: Why must AI predictions in drug discovery be experimentally validated?
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Why must AI predictions in drug discovery be experimentally validated?
assistant
AI predictions in drug discovery require experimental validation because they 

## 8. Save the LoRA adapter


In [11]:
ADAPTER_DIR = "./outputs/pharma-dpo-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved:", ADAPTER_DIR)


Saved: ./outputs/pharma-dpo-adapter


## Interpretation

This experiment demonstrates how the same DPO machinery behaves with a real pharmaceutical preference dataset. It does **not** establish biomedical factual accuracy. For a more scientifically meaningful experiment, use a licensed biomedical knowledge source to construct evidence-grounded preference pairs and evaluate factual correctness separately.
